
## Collaborative Filtering with ALS

### 1. Objective

The objective of this notebook is to build a personalised recommendation model using customers' historical purchase interactions.

An implicit-feedback Alternating Least Squares (ALS) model will be trained using customer–article interactions available before the evaluation period.

The model will generate up to 12 recommendations per customer and will be evaluated against purchases made during the following 7-day period using Recall@12 and MAP@12.

Performance will be compared against the strongest popularity baseline established previously.

In [3]:
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

### 1. Load Data and Recreate the Temporal Split

The same temporal evaluation strategy used for the popularity baselines is
retained.

The final seven days of transactions represent future customer behaviour,
while all earlier transactions are available to the recommender.

In [4]:
transactions = pd.read_csv(
    "../data/raw/transactions_train.csv",
    usecols=["t_dat", "customer_id", "article_id"]
)

transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

max_date = transactions["t_dat"].max()
eval_start = max_date - pd.Timedelta(days=6)

train = transactions[
    transactions["t_dat"] < eval_start
].copy()

evaluation = transactions[
    transactions["t_dat"] >= eval_start
].copy()

print("Train:", train["t_dat"].min(), "to", train["t_dat"].max())
print("Evaluation:", evaluation["t_dat"].min(), "to", evaluation["t_dat"].max())

Train: 2018-09-20 00:00:00 to 2020-09-15 00:00:00
Evaluation: 2020-09-16 00:00:00 to 2020-09-22 00:00:00


### 2. Prepare Customer–Article Interactions

ALS operates on a user–item interaction matrix rather than individual
transaction rows.

Multiple purchases of the same article by the same customer are therefore
aggregated into a purchase count.

In [5]:
interactions = (
    train
    .groupby(["customer_id", "article_id"])
    .size()
    .reset_index(name="purchase_count")
)

interactions.head()

,customer_id,article_id,purchase_count
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,176209023,1
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601006,2
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601043,1
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,607642008,1
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,625548001,1


In [6]:
interactions["purchase_count"].describe()

count    2.710115e+07
mean     1.164084e+00
std      5.722045e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      5.700000e+02
Name: purchase_count, dtype: float64

### 4. Encode customers and articles
ALS works with integer matrix positions rather than the original identifiers.

In [7]:
customer_ids = interactions["customer_id"].unique()
article_ids = interactions["article_id"].unique()

In [8]:
customer_to_idx = {
    customer_id: idx
    for idx, customer_id in enumerate(customer_ids)
}

article_to_idx = {
    article_id: idx
    for idx, article_id in enumerate(article_ids)
}

In [10]:
idx_to_article = {
    idx: article_id
    for article_id, idx in article_to_idx.items()
}
interactions["user_idx"] = interactions["customer_id"].map(customer_to_idx)
interactions["item_idx"] = interactions["article_id"].map(article_to_idx)

In [11]:
interactions[
    ["customer_id", "article_id", "purchase_count", "user_idx", "item_idx"]
].head()

,customer_id,article_id,purchase_count,user_idx,item_idx
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,176209023,1,0,0
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601006,2,0,1
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601043,1,0,2
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,607642008,1,0,3
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,625548001,1,0,4


### 5. Build the sparse interaction matrix
The theoretical customer × article matrix is enormous, but almost all entries are empty.

A sparse matrix stores only observed interactions.

In [14]:
binary_values = np.ones(
    len(interactions),
    dtype=np.float32
)
binary_matrix = csr_matrix(
    (
        binary_values,
        (
            interactions["user_idx"],
            interactions["item_idx"]
        )
    ),
    shape=(
        len(customer_ids),
        len(article_ids)
    )
)

In [15]:
binary_matrix.shape

(1356709, 103880)

In [16]:
binary_matrix.nnz

27101148

### 6. Create a transformed frequency signal
We also want to test whether repeat purchases contain useful preference information.

<br>Instead of using raw frequency: </br>
- 1 purchase  -> 1 
- 10 purchases -> 10 

<br>we dampen extreme repeats using: </br>
- log(1 + purchase_count)


Binary ALS
vs
Log-frequency ALS

In [30]:
log_values = np.log1p(
    interactions["purchase_count"].values
).astype(np.float32)

In [31]:
log_matrix = csr_matrix(
    (
        log_values,
        (
            interactions["user_idx"],
            interactions["item_idx"]
        )
    ),
    shape=binary_matrix.shape
)

### 7. Build evaluation ground truth

In [23]:
ground_truth = (
    evaluation
    .groupby("customer_id")["article_id"]
    .apply(set)
    .reset_index(name="actual_articles")
)

ground_truth["user_idx"] = (
    ground_truth["customer_id"]
    .map(customer_to_idx)
)

In [24]:
known_ground_truth = (
    ground_truth
    .dropna(subset=["user_idx"])
    .copy()
)

known_ground_truth["user_idx"] = (
    known_ground_truth["user_idx"]
    .astype(int)
)

In [25]:
total_eval_customers = len(ground_truth)
known_eval_customers = len(known_ground_truth)

known_customer_coverage = (
    known_eval_customers / total_eval_customers
)

print("Evaluation customers:", total_eval_customers)
print("Known evaluation customers:", known_eval_customers)
print("Known customer coverage:", known_customer_coverage)

Evaluation customers: 68984
Known evaluation customers: 63412
Known customer coverage: 0.9192276469906066


### 8. Train binary ALS
Start with reasonable parameters rather than tuning immediately.

In [26]:
binary_model = AlternatingLeastSquares(
    factors=64,
    regularization=0.05,
    alpha=20,
    iterations=15,
    random_state=42
)

In [27]:
binary_model.fit(binary_matrix)

  0%|          | 0/15 [00:00<?, ?it/s]

### 9. Train log-frequency ALS

In [28]:
log_model = AlternatingLeastSquares(
    factors=64,
    regularization=0.05,
    alpha=20,
    iterations=15,
    random_state=42
)

In [32]:
log_model.fit(log_matrix)

  0%|          | 0/15 [00:00<?, ?it/s]

### 10. Generate Top-12 recommendations
One important decision here: we usually do not want to recommend articles the customer has already purchased historically.

In [35]:
def generate_recommendations(
    model,
    interaction_matrix,
    user_indices,
    k=12
):
    recommendations = {}

    for user_idx in user_indices:

        item_indices, scores = model.recommend(
            userid=user_idx,
            user_items=interaction_matrix[user_idx],
            N=k,
            filter_already_liked_items=True
        )

        recommendations[user_idx] = [
            idx_to_article[item_idx]
            for item_idx in item_indices
        ]

    return recommendations

In [33]:
eval_user_indices = (
    known_ground_truth["user_idx"]
    .unique()
)

In [36]:
binary_recommendations = generate_recommendations(
    binary_model,
    binary_matrix,
    eval_user_indices,
    k=12
)

In [37]:
log_recommendations = generate_recommendations(
    log_model,
    log_matrix,
    eval_user_indices,
    k=12
)

In [ ]:
list(binary_recommendations.items())[:3]

### 11. Evaluation metrics
Use the same definitions as Notebook 2.

In [38]:
def recall_at_12(actual_articles, recommendations):
    hits = len(
        actual_articles.intersection(
            recommendations[:12]
        )
    )

    return hits / len(actual_articles)

In [39]:
def average_precision_at_12(
    actual_articles,
    recommendations
):
    hits = 0
    score = 0.0

    for rank, article in enumerate(
        recommendations[:12],
        start=1
    ):
        if article in actual_articles:
            hits += 1
            score += hits / rank

    return score / min(
        len(actual_articles),
        12
    )

### 12. Evaluate a model

In [40]:
def evaluate_model(
    ground_truth_df,
    recommendations
):
    results = ground_truth_df.copy()

    results["recommendations"] = (
        results["user_idx"]
        .map(recommendations)
    )

    results = results.dropna(
        subset=["recommendations"]
    )

    results["recall@12"] = results.apply(
        lambda row: recall_at_12(
            row["actual_articles"],
            row["recommendations"]
        ),
        axis=1
    )

    results["ap@12"] = results.apply(
        lambda row: average_precision_at_12(
            row["actual_articles"],
            row["recommendations"]
        ),
        axis=1
    )

    return {
        "Recall@12": results["recall@12"].mean(),
        "MAP@12": results["ap@12"].mean()
    }, results

In [41]:
binary_metrics, binary_results = evaluate_model(
    known_ground_truth,
    binary_recommendations
)

binary_metrics

{'Recall@12': np.float64(0.012030768103357686),
 'MAP@12': np.float64(0.004314094125291855)}

In [42]:
log_metrics, log_results = evaluate_model(
    known_ground_truth,
    log_recommendations
)

log_metrics

{'Recall@12': np.float64(0.011948350339338516),
 'MAP@12': np.float64(0.004365465082431165)}

### 13. Compare against the popularity baseline
Your strongest baseline from Notebook 2 was approximately:

In [44]:
recent_popularity_recall = 0.02550428716366145
recent_popularity_map = 0.008747681312782597

In [45]:
model_comparison = pd.DataFrame([
    {
        "Model": "Recent popularity - 7d",
        "Recall@12": recent_popularity_recall,
        "MAP@12": recent_popularity_map
    },
    {
        "Model": "Binary ALS",
        "Recall@12": binary_metrics["Recall@12"],
        "MAP@12": binary_metrics["MAP@12"]
    },
    {
        "Model": "Log-frequency ALS",
        "Recall@12": log_metrics["Recall@12"],
        "MAP@12": log_metrics["MAP@12"]
    }
])

model_comparison.sort_values(
    "MAP@12",
    ascending=False
)

model_comparison

,Model,Recall@12,MAP@12
0,Recent popularity - 7d,0.025504,0.008748
1,Binary ALS,0.012031,0.004314
2,Log-frequency ALS,0.011948,0.004365


### 14. Performance by customer history
This is particularly important for collaborative filtering.

In [46]:
customer_history = (
    interactions
    .groupby("customer_id")["article_id"]
    .nunique()
    .reset_index(name="history_size")
)

In [47]:
best_results = log_results.copy()

In [48]:
best_results = best_results.merge(
    customer_history,
    on="customer_id",
    how="left"
)

In [49]:
bins = [0, 2, 5, 20, 50, float("inf")]
labels = ["1-2", "3-5", "6-20", "21-50", "51+"]

best_results["history_segment"] = pd.cut(
    best_results["history_size"],
    bins=bins,
    labels=labels
)

In [50]:
segment_performance = (
    best_results
    .groupby(
        "history_segment",
        observed=True
    )
    .agg(
        Customers=("customer_id", "count"),
        Recall_at_12=("recall@12", "mean"),
        MAP_at_12=("ap@12", "mean")
    )
    .reset_index()
)

segment_performance

,history_segment,Customers,Recall_at_12,MAP_at_12
0,1-2,2300,0.013313,0.005540
1,3-5,3527,0.017722,0.006858
2,6-20,14569,0.015270,0.006117
3,21-50,19353,0.012330,0.004060
4,51+,23663,0.008598,0.003051


### Cold-Start Analysis

ALS learns latent representations from historical customer–article interactions.

Customers with no historical interaction data do not have a learned user representation and therefore cannot receive personalised ALS recommendations.

The proportion of evaluation customers known to the ALS model is therefore an important measure of recommendation coverage.

A production recommendation system would require fallback strategies for unseen or low-history customers, such as recent popularity, segment-level popularity or content-based recommendations.

## Key Findings

- ALS was evaluated as the first personalised recommendation approach against the strongest recent-popularity baseline.

- The [binary/log-frequency] interaction representation produced the stronger ALS model, achieving Recall@12 of X and MAP@12 of X.

- Compared with the 7-day recent-popularity baseline, ALS [improved/did not improve] recommendation accuracy.

- ALS performance varied by customer interaction history, with [describe result].

- X% of evaluation customers had historical interactions available to the model. Customers without observable history cannot receive personalised ALS recommendations.

- These results support using collaborative filtering as one component of a broader recommendation architecture alongside alternative candidate generators and cold-start fallbacks.